# 🚀 LoRA Adapter Packaging Pipeline

This pipeline locates a LoRA adapter from Kaggle model inputs, validates its
structure and rank, and packages it into a submission-ready zip.

---

## 🔄 Workflow Overview

```mermaid
flowchart TD
    A[Define ADAPTER_HINTS] --> B[Auto-discover adapter directory]
    B --> C{Found?}
    C -- No --> D[Dump /kaggle/input tree & abort]
    C -- Yes --> E[Validate adapter_config.json]
    E --> F{Rank <= 32?}
    F -- No --> G[Abort — rank too high]
    F -- Yes --> H[Report file sizes]
    H --> I[Create submission.zip at root level]
    I --> J[Verify zip contents]
    J --> K[Preview new reasoning-trace dataset]
    K --> L[Final Output Ready]
```

---

## 📦 Step-by-Step Breakdown

### 1️⃣ Configure Hints

Two known Kaggle model paths are listed in `ADAPTER_HINTS`. The script tries
each in order — no manual path-fixing needed in most cases.

```python
ADAPTER_HINTS = [
    "/kaggle/input/models/huikang/nemotron-adapter",
    "/kaggle/input/models/kienngx/nemotron-nano-30b-trained",
]
```

---

### 2️⃣ Auto-discover Adapter Directory

`find_adapter_dir()` walks up to 5 levels deep inside each hint (handling
Kaggle Hub version subdirectories like `transformers/default/20/`) and returns
the first folder that contains **both** required files.

If nothing is found, the full `/kaggle/input` tree is printed with `← ✓`
markers on any line that matches a required file, making the correct path
immediately obvious.

---

### 3️⃣ Validate `adapter_config.json`

Reads the config and checks:

| field | check |
|---|---|
| `base_model_name_or_path` | printed for inspection |
| `peft_type` | printed for inspection |
| `r` / `lora_rank` | must be **≤ 32** (competition limit) |


Aborts with a clear message if the rank exceeds the limit.

---

### 4️⃣ Report File Sizes

```
adapter directory : /kaggle/input/models/huikang/nemotron-adapter/...
────────────────────────────────────────────────────
  ✓ adapter_config.json                       0.0 MB
  ✓ adapter_model.safetensors              3389.7 MB
```

Any extra files in the folder are listed but skipped — only the two required
files enter the zip.

---

### 5️⃣ Packaging Step 📦

Creates `submission.zip` with a **flat structure** (no subdirectory prefix),
which is required by the competition evaluator.

```python
zf.write(adapter_dir / fname, arcname=fname)   # flat — no subdir
```

---

### 6️⃣ Validation Checks ✅

Before finishing, the script confirms:

- Both files exist at the **zip root** (not inside a subfolder)
- `adapter_config.json` is readable JSON inside the zip
- Rank is re-confirmed from the in-zip copy

---

### 7️⃣ New Dataset Preview

If the reasoning-trace dataset is attached, it is automatically previewed:

```
new dataset  (ex.csv)
────────────────────────────────────────────────────
  rows    : 1
  columns : ['id', 'prompt', 'answer', 'reasoning_trace']
  ✓ reasoning_trace column present — ready for CoT fine-tuning
```

---

## 📊 Data Flow Summary

```mermaid
flowchart LR
    A[ADAPTER_HINTS] --> B[find_adapter_dir — 5-level walk]
    B --> C[validate_adapter_config — rank check]
    C --> D[build_and_verify_zip — flat arcname]
    D --> E[submission.zip]
    E --> F[Integrity check from inside zip]
    F --> G[Ready for Kaggle submission]
```

---

## 🧠 Key Improvements Over Original

| area | before | after |
|---|---|---|
| path resolution | hard-coded, breaks silently | auto-discovery with fallback tree dump |
| rank validation | not checked | aborts if rank > 32 |
| zip structure | subdirectory possible | always flat at root |
| config re-check | from disk only | re-read from inside the zip |
| new dataset | ignored | auto-previewed if attached |
| error messages | generic FileNotFoundError | annotated tree shows exact missing file |

---

## 📌 Final Output

```
submission.zip
```

✔ `adapter_config.json` at zip root  
✔ `adapter_model.safetensors` at zip root  
✔ Rank ≤ 32 confirmed  
✔ Config readable from inside the zip  
✔ Deployment-ready format  

---


In [ ]:
import os, zipfile, json

ADAPTER_DIR = '/kaggle/input/datasets/assiabenazzouz/adappter-v32-epoch-5/adapter'
zip_path = '/kaggle/working/submission.zip'

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in ["adapter_config.json", "adapter_model.safetensors"]:
        fpath = os.path.join(ADAPTER_DIR, fname)
        if os.path.exists(fpath):
            zf.write(fpath, fname)
            print(f"✓ Added: {fname}")
        else:
            print(f"× Missing: {fname}")

size = os.path.getsize(zip_path)/1024/1024
print(f"submission.zip: {size:.1f} MB")

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("assiabenazzouz/adappter-v32-epoch-5")

print("Path to dataset files:", path)